# 05 — GenAI Integration (Agentic AI with Tool Calling)
**Merchant Network Analytics — Capstone Project BNI ODP Data Analytics**

---

## Tujuan Notebook Ini
Membangun AI agent yang bisa menjawab pertanyaan tentang target akuisisi merchant dalam bahasa natural.

**Arsitektur: Tool Calling (Level Agentic)**
- LLM menerima pertanyaan user + daftar tools yang tersedia
- LLM **memutuskan sendiri** tool mana yang perlu dipanggil
- Tool dipanggil → hasilnya dikirim balik ke LLM
- LLM merumuskan jawaban akhir berdasarkan data dari tools

**Bukan**: basic prompt template atau RAG biasa.  
**Adalah**: agentic workflow di mana LLM bertindak sebagai "otak" yang mengorkestrasi tools.

**Input:** Semua artifacts dari notebook 01-04  
**Output:** `agent_config.pkl` (konfigurasi agent untuk dashboard)

## 1. Load Artifacts

In [1]:
import pandas as pd
import numpy as np
import pickle
import joblib
import json
import os
from dotenv import load_dotenv

load_dotenv()

import os
PROJECT_ROOT = '.' if os.path.isdir('./data') else '..'

# PASTIKAN ini yang dibaca (bukan merchants_with_predictions.csv):
merchants_pred = pd.read_csv(f'{PROJECT_ROOT}/data/processed/merchants_with_product_rec.csv')

# Verifikasi kolom yang dibutuhkan ada
required_cols = ['merchant_id', 'nama', 'is_bni_acquiring', 'priority_score',
                 'priority_rank', 'priority_category', 'product_recommendation',
                 'product_rec_proba_edc']
missing_cols = [c for c in required_cols if c not in merchants_pred.columns]
if missing_cols:
    print(f"⚠️  Kolom yang hilang: {missing_cols}")
    print("   Pastikan notebook 03 dan 04 sudah dijalankan ulang")
else:
    print(f"✓ Semua kolom tersedia: {len(merchants_pred)} merchant")

affinity_edge_table = pd.read_csv(f'{PROJECT_ROOT}/data/processed/affinity_edge_table.csv')
merchants_clean = pd.read_csv(f'{PROJECT_ROOT}/data/processed/merchants_clean.csv')

with open(f'{PROJECT_ROOT}/models/graph_object.pkl', 'rb') as f:
    G = pickle.load(f)
with open(f'{PROJECT_ROOT}/models/community_map.pkl', 'rb') as f:
    community_map = pickle.load(f)

# Priority scoring sekarang composite score (notebook 03), bukan trained classifier —
# tidak ada lagi model_metadata.pkl / best_model.pkl untuk acquisition priority.
priority_weights = joblib.load(f'{PROJECT_ROOT}/models/priority_weights.pkl')

try:
    product_rec_model = joblib.load(f'{PROJECT_ROOT}/models/product_rec_model.pkl')
    prod_label_encoders_loaded = joblib.load(f'{PROJECT_ROOT}/models/product_rec_label_encoders.pkl')
    prod_feature_cols = joblib.load(f'{PROJECT_ROOT}/models/product_rec_feature_cols.pkl')
    product_rec_metadata = joblib.load(f'{PROJECT_ROOT}/models/product_rec_metadata.pkl')
    merchants_full = merchants_pred.copy()
    USE_PRODUCT_REC_MODEL = True
    print("Product recommendation model loaded.")
except FileNotFoundError:
    product_rec_model = None
    prod_label_encoders_loaded = {}
    prod_feature_cols = []
    product_rec_metadata = {}
    merchants_full = merchants_pred.copy()
    USE_PRODUCT_REC_MODEL = False
    print("Product rec model not found, using rule-based fallback.")

print(f"Merchants: {len(merchants_pred)}")
print(f"Affinity edges: {len(affinity_edge_table)}")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (undirected={not G.is_directed()})")
print(f"Priority scoring weights: {priority_weights}")
print(f"Product rec model: {product_rec_metadata.get('model_name', '-')}")

✓ Semua kolom tersedia: 708 merchant
Product recommendation model loaded.
Merchants: 708
Affinity edges: 1125
Graph: 627 nodes, 1125 edges (undirected=True)
Priority scoring weights: {'pagerank': 0.3, 'connected_bni_ratio': 0.25, 'degree': 0.25, 'weighted_degree': 0.2}
Product rec model: Logistic Regression


## 2. Definisi Tool Functions

Setiap function adalah satu "tool" yang bisa dipanggil oleh LLM.
LLM tidak menjalankan Python secara langsung — LLM membaca deskripsi tool dan memutuskan
tool mana yang relevan untuk menjawab pertanyaan user.

In [2]:
def get_top_acquisition_targets(n: int = 5, kategori: str = None, kota: str = None) -> str:
    """Ambil daftar merchant prioritas akuisisi tertinggi."""
    df = merchants_full[merchants_full['is_bni_acquiring'] == 'Tidak'].copy()
    if kategori:
        df = df[df['kategori'].str.lower() == kategori.lower()]
    if kota:
        df = df[df['kota'].str.lower() == kota.lower()]
    if len(df) == 0:
        return json.dumps({"error": "Tidak ada merchant yang sesuai filter"})

    top = df.nlargest(n, 'priority_score')
    results = []
    for _, row in top.iterrows():
        results.append({
            "merchant_id"         : row['merchant_id'],
            "nama"                : row.get('nama', 'Unknown'),
            "kategori"            : row['kategori'],
            "kota"                : row.get('kota', 'Unknown'),
            "priority_score"      : round(float(row.get('priority_score', 0)), 4),
            "priority_category"   : row.get('priority_category', 'Tidak Teranalisis'),
            "degree"              : int(row.get('degree', 0)),
            "connected_bni_ratio" : round(float(row.get('connected_bni_ratio', 0)), 3),
            "product_recommendation": row.get('product_recommendation', 'QRIS'),
        })
    return json.dumps({"targets": results, "total_found": len(df)}, ensure_ascii=False)


def search_merchant_by_name(nama: str) -> str:
    """Cari ID merchant berdasarkan namanya (pencarian teks parsial)."""
    # Mencari nama yang mengandung kata kunci (case-insensitive)
    matches = merchants_full[merchants_full['nama'].str.contains(nama, case=False, na=False)]
    
    if len(matches) == 0:
        return json.dumps({"error": f"Merchant dengan nama '{nama}' tidak ditemukan."})
    
    # Ambil maksimal 5 hasil teratas agar prompt AI tidak kepenuhan
    results = []
    for _, row in matches.head(5).iterrows():
        results.append({
            "merchant_id": row['merchant_id'],
            "nama": row['nama'],
            "kategori": row['kategori'],
            "kota": row.get('kota', '?')
        })
        
    return json.dumps({
        "info": f"Ditemukan {len(matches)} hasil pencarian.",
        "matches": results
    }, ensure_ascii=False)
    
def get_merchant_network(merchant_id: str) -> str:
    """Ambil detail jaringan afinitas pelanggan satu merchant."""
    if merchant_id not in G.nodes():
        return json.dumps({"error": f"Merchant {merchant_id} tidak ditemukan di graph"})

    def enrich(mid):
        info = merchants_full[merchants_full['merchant_id'] == mid]
        if len(info) > 0:
            row = info.iloc[0]
            return {
                "merchant_id": mid,
                "nama": row.get('nama', '?'),
                "kategori": row['kategori'],
                "is_bni_acquiring": row['is_bni_acquiring'],
            }
        return {"merchant_id": mid}

    neighbors = []
    for neighbor in G.neighbors(merchant_id):
        edge = G.edges[merchant_id, neighbor]
        neighbors.append({
            **enrich(neighbor),
            "shared_customers": int(edge.get('shared_customers', 0)),
            "jaccard": float(edge.get('jaccard', 0)),
        })
    neighbors.sort(key=lambda item: item['jaccard'], reverse=True)

    m_info = merchants_full[merchants_full['merchant_id'] == merchant_id]
    own = {}
    if len(m_info) > 0:
        row = m_info.iloc[0]
        own = {
            "merchant_id": merchant_id,
            "nama": row.get('nama', '?'),
            "kategori": row['kategori'],
            "kota": row.get('kota', '?'),
            "is_bni_acquiring": row['is_bni_acquiring'],
            "degree": int(row.get('degree', 0)),
            "weighted_degree": float(row.get('weighted_degree', 0)),
            "priority_score": round(float(row.get('priority_score', 0)), 4),
        }

    return json.dumps(
        {"merchant": own, "affinity_neighbors": neighbors, "total_neighbors": len(neighbors)},
        ensure_ascii=False,
        default=str,
    )


def calculate_acquisition_impact(merchant_id: str) -> str:
    """Estimasi dampak finansial akuisisi dan rekomendasi produk ML."""
    if merchant_id not in G.nodes():
        return json.dumps({"error": f"Merchant {merchant_id} tidak ditemukan"})

    m_info = merchants_full[merchants_full['merchant_id'] == merchant_id]
    if len(m_info) == 0:
        return json.dumps({"error": "Data merchant tidak ditemukan"})

    row = m_info.iloc[0]
    # Affinity graph tidak menyimpan amount. Omzet bulanan menjadi basis estimasi MDR.
    monthly_flow = float(row.get('avg_omzet_bulanan', 0) or 0)
    fee_rate = 0.007
    monthly_fee = monthly_flow * fee_rate

    neighbors = set(G.neighbors(merchant_id))
    bni_set = set(merchants_full[merchants_full['is_bni_acquiring'] == 'Ya']['merchant_id'])
    bni_neighbors = neighbors & bni_set

    # Cari product recommendation dari ML model (notebook 04)
    prod_info = merchants_pred[merchants_pred['merchant_id'] == merchant_id]
    if len(prod_info) > 0 and 'product_recommendation' in prod_info.columns:
        rec_produk = prod_info.iloc[0].get('product_recommendation', None)
        rec_confidence = prod_info.iloc[0].get('product_rec_proba_edc', 0.5)
        rec_source = prod_info.iloc[0].get('product_rec_source', 'unknown')

        if pd.notna(rec_produk):
            if rec_produk == 'QRIS + EDC':
                layanan_text = f"QRIS + EDC (confidence: {rec_confidence:.0%})"
            else:
                layanan_text = f"QRIS (confidence: {1-rec_confidence:.0%})"
        else:
            # Fallback jika tidak ada prediksi
            layanan_text = "QRIS (rekomendasi awal)"
    else:
        layanan_text = "QRIS (data tidak tersedia)"
        rec_source = "unknown"

    return json.dumps({
        "merchant_id": merchant_id,
        "nama": row.get('nama', '?'),
        "monthly_avg_flow": monthly_flow,
        "est_monthly_fee_income": float(monthly_fee),
        "est_annual_fee_income": float(monthly_fee * 12),
        "fee_rate": fee_rate,
        "connected_bni_merchants": len(bni_neighbors),
        "total_connected": len(neighbors),
        "rekomendasi_layanan": layanan_text,
        "recommendation_source": rec_source,
        "network_benefit": f"Akuisisi menghubungkan {len(bni_neighbors)} merchant BNI dalam affinity graph",
    }, ensure_ascii=False, default=str)


def get_overview_stats() -> str:
    """Statistik keseluruhan jaringan merchant."""
    total = len(merchants_full)
    bni = (merchants_full['is_bni_acquiring'] == 'Ya').sum()
    hp = (merchants_full.get('predicted_priority', pd.Series([0])) == 1).sum()

    return json.dumps({
        "total_merchants": int(total),
        "bni_customers": int(bni),
        "non_bni": int(total - bni),
        "bni_penetration": round(float(bni / total * 100), 1),
        "high_priority": int(hp),
        "total_affinity_edges": int(G.number_of_edges()),
        "communities": len(set(community_map.values())),
        "model_performance": product_rec_metadata.get('test_metrics', {}),
        "product_rec_model_active": USE_PRODUCT_REC_MODEL,
    }, ensure_ascii=False, default=str)


TOOLS_REGISTRY = {
    "search_merchant_by_name": search_merchant_by_name,
    "get_top_acquisition_targets": get_top_acquisition_targets,
    "get_merchant_network": get_merchant_network,
    "calculate_acquisition_impact": calculate_acquisition_impact,
    "get_overview_stats": get_overview_stats,
}
print(f"Registered {len(TOOLS_REGISTRY)} tools:")
for name in TOOLS_REGISTRY:
    print(f"  - {name}")

Registered 5 tools:
  - search_merchant_by_name
  - get_top_acquisition_targets
  - get_merchant_network
  - calculate_acquisition_impact
  - get_overview_stats


## 3. Tool Schemas (Format untuk LLM API)

In [3]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_top_acquisition_targets",
            "description": "Ambil daftar merchant prioritas akuisisi tertinggi. Bisa difilter per kategori dan/atau kota.",
            "parameters": {
                "type": "object",
                "properties": {
                    "n"        : {"type": "integer", "description": "Jumlah merchant (default 5)"},
                    "kategori" : {"type": "string", "description": "Filter kategori: F&B, Fashion, Grocery, Electronics, Automotive"},
                    "kota"     : {"type": "string", "description": "Filter kota: Jakarta, Bogor, Bandung, Tangerang, Bekasi, Depok"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_merchant_network",
            "description": "Detail jaringan afinitas satu merchant — tetangga dengan customer overlap dan Jaccard tertinggi.",
            "parameters": {
                "type": "object",
                "properties": {
                    "merchant_id": {"type": "string", "description": "ID merchant (contoh: M0001)"}
                },
                "required": ["merchant_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_acquisition_impact",
            "description": "Estimasi dampak finansial jika merchant diakuisisi serta rekomendasi produk dari model ML.",
            "parameters": {
                "type": "object",
                "properties": {
                    "merchant_id": {"type": "string", "description": "ID merchant"}
                },
                "required": ["merchant_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_overview_stats",
            "description": "Statistik keseluruhan jaringan: total merchant, penetrasi BNI, performa model.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
        {
        "type": "function",
        "function": {
            "name": "search_merchant_by_name",
            "description": "Cari ID merchant berdasarkan namanya. Gunakan ini TERLEBIH DAHULU jika user bertanya menggunakan nama merchant tanpa menyebutkan ID-nya.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nama": {"type": "string", "description": "Nama merchant yang ingin dicari (contoh: 'Tekstil Pratama')"}
                },
                "required": ["nama"]
            }
        }
    },

]
print(f"Defined {len(TOOL_SCHEMAS)} tool schemas")

Defined 5 tool schemas


## 4. Agent Function

In [4]:
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama") if OpenAI is not None else None

SYSTEM_PROMPT = """Kamu adalah Asisten Akuisisi Merchant untuk tim Sales Bank BNI.
Tugasmu membantu tim mengidentifikasi merchant prioritas untuk diakuisisi berdasarkan customer-merchant affinity graph.

Panduan:
- Selalu gunakan tools untuk menjawab — jangan mengarang data
- Jawab dalam Bahasa Indonesia yang profesional
- Sertakan angka konkret (estimasi fee income, jumlah koneksi)
- Gunakan rekomendasi produk ML (QRIS atau QRIS + EDC) dari tool
- Jelaskan MENGAPA merchant diprioritaskan berdasarkan posisi di affinity graph
"""

def run_agent(user_message: str, max_turns: int = 5) -> str:
    """Jalankan agentic tool-calling loop menggunakan OpenAI compatibility (Ollama lokal)."""
    if client is None:
        raise RuntimeError("OpenAI package belum terinstall.")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    for _ in range(max_turns):
        response = client.chat.completions.create(
            model="gemma4:e2b",
            messages=messages,
            tools=TOOL_SCHEMAS,
            temperature=0.0
        )
        
        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                import json
                try:
                    tool_input = json.loads(tool_call.function.arguments)
                except Exception:
                    tool_input = {}
                    
                print(f"  [Tool dipanggil] {tool_name}({tool_input})")
                if tool_name in TOOLS_REGISTRY:
                    result = TOOLS_REGISTRY[tool_name](**tool_input)
                else:
                    result = json.dumps({"error": f"Tool {tool_name} tidak ditemukan"})
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": result,
                })
        else:
            return message.content
            
    return "Maaf, tidak dapat menyelesaikan analisis dalam batas iterasi."

print("Agent function siap." if client is not None else "Agent function tidak siap.")

Agent function tidak siap.


## 5. Test Agent

In [5]:
# Test 1: Overview
print("=" * 60)
print("TEST 1: Overview Jaringan")
print("=" * 60)
try:
    response = run_agent("Berikan ringkasan kondisi jaringan merchant kita saat ini.")
    print(response)
except Exception as e:
    print(f"Error: {e}")
    print("Pastikan Ollama sudah berjalan dan model 'gemma4:e2b' tersedia di lokal")

TEST 1: Overview Jaringan
Error: OpenAI package belum terinstall.
Pastikan Ollama sudah berjalan dan model 'gemma4:e2b' tersedia di lokal


In [6]:
# Test 2: Top targets dengan filter
print("\n" + "=" * 60)
print("TEST 2: Target Akuisisi F&B Jakarta")
print("=" * 60)
try:
    response = run_agent("Merchant mana yang harus diprioritaskan untuk akuisisi di Jakarta, kategori F&B?")
    print(response)
except Exception as e:
    print(f"Error: {e}")


TEST 2: Target Akuisisi F&B Jakarta
Error: OpenAI package belum terinstall.


In [7]:
# Test 3: Deep dive merchant spesifik
print("\n" + "=" * 60)
print("TEST 3: Analisis Merchant Spesifik")
print("=" * 60)
top_target = merchants_pred[merchants_pred['is_bni_acquiring'] == 'Tidak'].nlargest(1, 'priority_score')
if len(top_target) > 0:
    target_id   = top_target.iloc[0]['merchant_id']
    target_nama = top_target.iloc[0].get('nama', 'Unknown')
    try:
        response = run_agent(
            f"Jelaskan mengapa {target_nama} ({target_id}) menjadi target akuisisi prioritas. "
            f"Berapa estimasi fee income-nya?"
        )
        print(response)
    except Exception as e:
        print(f"Error: {e}")


TEST 3: Analisis Merchant Spesifik
Error: OpenAI package belum terinstall.


## 6. Simpan Agent Config

In [8]:
agent_config = {
    "system_prompt": SYSTEM_PROMPT,
    "tool_schemas" : TOOL_SCHEMAS,
    "model"        : "gemma4:e2b"
}
joblib.dump(agent_config, f'{PROJECT_ROOT}/models/agent_config.pkl')
print("Saved: agent_config.pkl")
print("\nAgent siap diintegrasikan ke dashboard (06_dashboard.py)")
print("\nContoh pertanyaan yang bisa dijawab agent:")
questions = [
    "Top 5 target akuisisi di Bogor",
    "Jelaskan jaringan koneksi merchant M0042",
    "Estimasi fee income kalau kita akuisisi hub terbesar F&B",
    "Ekosistem mana yang penetrasi BNI-nya paling rendah?",
    "Rekomendasikan 3 merchant untuk dikunjungi RM besok"
]
for q in questions:
    print(f"  • {q}")

Saved: agent_config.pkl

Agent siap diintegrasikan ke dashboard (06_dashboard.py)

Contoh pertanyaan yang bisa dijawab agent:
  • Top 5 target akuisisi di Bogor
  • Jelaskan jaringan koneksi merchant M0042
  • Estimasi fee income kalau kita akuisisi hub terbesar F&B
  • Ekosistem mana yang penetrasi BNI-nya paling rendah?
  • Rekomendasikan 3 merchant untuk dikunjungi RM besok
